<a href="https://colab.research.google.com/github/agil-p-varghese/NanoGPT-implementation/blob/main/Decoder_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import math
import requests
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F

In [ ]:
#Downloading and prepare tiny shakesphere data (char level)
data_url='https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
if not os.path.exists('input.txt'):
  with open('input.txt','w',encoding='utf-8') as f:
    f.write(requests.get(data_url).text)

with open('input.txt','r',encoding='utf-8') as f:
  text=f.read()

#Build vocabulary::unique characters in text
chars=sorted(list(set(text)))
vocab_size=len(chars)
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for i,ch in enumerate(chars)}
encode=lambda s:[stoi[c] for c in s]
decode=lambda l:''.join([itos[i] for i in l])

#split into train(90%) and validation(10%) set
data=torch.tensor(encode(text),dtype=torch.long)
n=int(0.9*len(data))
train_data=data[:n]
val_data=data[n:]

In [ ]:
##2.HyperParamters

batch_size=64 #sequence run  in paralell
block_size=256 #max context length for prediction
max_iters=8000#total training steps
eval_interval=250#evaluation frequency
learning_rate=1e-3
device='cuda' if torch.cuda.is_available() else 'cpu'
eval_iters=200
n_embd=384
n_head=6
n_layer=6
dropout=0.2

print(f"using device::{device} | vocab_size::{vocab_size}")



using device::cuda | vocab_size::65


In [ ]:
#Model archietechture(Layer Norm,causal self attention,mlp,gpt)

class LayerNorm(nn.Module):
  def __init__(self,ndim,bias=True):
    super().__init__()
    self.weight=nn.Parameter(torch.ones(ndim))
    self.bias=nn.Parameter(torch.zeros(ndim)) if bias else None

  def forward(self,input):
    return F.layer_norm(input,self.weight.shape,self.weight,self.bias,1e-5)

class CausalSelfAttention(nn.Module):
  def __init__(self,n_embd,n_head,block_size,dropout):
    super().__init__()
    assert n_embd % n_head==0
    self.c_attn=nn.Linear(n_embd,3* n_embd)
    self.c_proj=nn.Linear(n_embd,n_embd)
    self.attn_dropout=nn.Dropout(dropout)
    self.resid_dropout=nn.Dropout(dropout)
    self.n_head=n_head
    self.n_embd=n_embd
    self.dropout=dropout
    self.flash=hasattr(torch.nn.functional,'scaled_dot_product_attention')
    if not self.flash:
      self.register_buffer("bias",torch.tril(torch.ones(block_size,block_size)).view(1,1,block_size,block_size))

  def forward(self,x):
    B,T,C=x.size()
    q,k,v=self.c_attn(x).split(self.n_embd,dim=2)
    k=k.view(B,T,self.n_head,C // self.n_head).transpose(1,2)
    q=q.view(B,T,self.n_head,C // self.n_head).transpose(1,2)
    v=v.view(B,T,self.n_head,C // self.n_head).transpose(1,2)

    if self.flash:
      y=F.scaled_dot_product_attention(q,k,v,attn_mask=None,dropout_p=self.dropout if self.training else 0,is_causal=True)

    else:
      att=(q @ k.transpose(-2,-1)) * (1.0/math.sqrt(k.size(-1)))
      att=att.masked_fill(self.bias[:,:,:T,:T]==0,float('-inf'))
      att=F.softmax(att,dim=-1)
      att=self.attn_dropout(att)
      y= att @ v

    y=y.transpose(1,2).contiguous().view(B,T,C)
    return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
  def __init__(self,n_embd,dropout):
    super().__init__()
    self.c_fc=nn.Linear(n_embd,4*n_embd)
    self.gelu=nn.GELU()
    self.c_proj=nn.Linear(4*n_embd,n_embd)
    self.dropout=nn.Dropout(dropout)

  def forward(self,x):
    return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
  def __init__(self,n_embd,n_head,block_size,dropout):
    super().__init__()
    self.ln_1=LayerNorm(n_embd)
    self.attn=CausalSelfAttention(n_embd,n_head,block_size,dropout)
    self.ln_2=LayerNorm(n_embd)
    self.mlp=MLP(n_embd,dropout)

  def forward(self,x):
    x=x+self.attn(self.ln_1(x))
    x=x+self.mlp(self.ln_2(x))
    return x
class NanoGPT(nn.Module):
  def __init__(self,vocab_size,n_embd,n_head,n_layer,block_size,dropout):
    super().__init__()
    self.block_size=block_size
    self.transformer=nn.ModuleDict(dict(
        wte=nn.Embedding(vocab_size,n_embd),
        wpe=nn.Embedding(block_size,n_embd),
        drop=nn.Dropout(dropout),
        h=nn.ModuleList([Block(n_embd,n_head,block_size,dropout) for _ in range(n_layer)]),
        ln_f=LayerNorm(n_embd),
    ))
    self.lm_head=nn.Linear(n_embd,vocab_size,bias=False)
    self.transformer.wte.weight=self.lm_head.weight #weight tying
  def forward(self,idx,targets=None):
    device=idx.device
    b,t=idx.size()
    assert t<=self.block_size, f"Cannot forward Sequence lenght {t} > max_context {self.block_size} "

    pos=torch.arange(0,t,dtype=torch.long,device=device)
    tok_emb=self.transformer.wte(idx)
    pos_emb=self.transformer.wpe(pos)
    x=self.transformer.drop(tok_emb+pos_emb)

    for block in self.transformer.h:
      x=block(x)
    x=self.transformer.ln_f(x)

    if targets is not None:
      logits=self.lm_head(x)
      loss=F.cross_entropy(logits.view(-1,logits.size(-1)),targets.view(-1))
    else:
      logits=self.lm_head(x[:,[-1],:])
      loss=None
    return logits,loss

  @torch.no_grad()
  def generate(self,idx,max_new_tokens,temperature=0.4,top_k=None):
    for _ in range(max_new_tokens):
      idx_cond=idx if idx.size(1)<= self.block_size else idx[:,-self.block_size:]
      logits,_=self(idx_cond)
      logits=logits[:,-1,:] / temperature
      if top_k is not None:
        v,_=torch.topk(logits,min(top_k,logits.size(-1)))
        logits[logits< v[:,[-1]]]=-float('Inf')
      probs=F.softmax(logits,dim=-1)
      idx_next=torch.multinomial(probs,num_samples=1)
      idx=torch.cat((idx,idx_next),dim=1)
    return idx

In [ ]:
#4.Helper Functions
def get_batch(split):
  dataset=train_data if split=='train' else val_data
  ix=torch.randint(len(dataset)-block_size,(batch_size,))
  x=torch.stack([dataset[i:i+block_size] for i in ix])
  y=torch.stack([dataset[i+1:i+block_size+1] for i in ix])
  return x.to(device),y.to(device)

@torch.no_grad()
def estimate_loss(model):
  out={}
  model.eval()
  for split in ['train','val']:
    losses=torch.zeros(eval_iters)
    for k in range(eval_iters):
      X,Y=get_batch(split)
      _,loss=model(X,Y)
      losses[k]=loss.item()
    out[split]=losses.mean()
  model.train()
  return out


In [ ]:
#Initialization and training loop
model=NanoGPT(vocab_size,n_embd,n_head,n_layer,block_size,dropout).to(device)
optimizer=torch.optim.AdamW(model.parameters(),lr=learning_rate)

print(f"total model parameters :{sum(p.numel() for p in model.parameters())/1e6:.2f}M")
for iter in range(max_iters):
  if iter% eval_interval==0 or iter==max_iters-1:
    losses=estimate_loss(model)
    print(f"step {iter}: train loss {losses['train']:.4f} , val_loss {losses['val']:.4f} ")

  xb,yb=get_batch('train')
  logits,loss=model(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()


total model parameters :10.77M
step 0: train loss 4.3567 , val_loss 4.3588 
step 250: train loss 3.3017 , val_loss 3.3359 
step 500: train loss 2.6286 , val_loss 2.6197 
step 750: train loss 2.0880 , val_loss 2.1086 
step 1000: train loss 1.7214 , val_loss 1.8563 
step 1250: train loss 1.5208 , val_loss 1.7118 
step 1500: train loss 1.4066 , val_loss 1.6117 
step 1750: train loss 1.3359 , val_loss 1.5589 
step 2000: train loss 1.2791 , val_loss 1.5186 
step 2250: train loss 1.2356 , val_loss 1.5083 
step 2500: train loss 1.2008 , val_loss 1.4950 
step 2750: train loss 1.1689 , val_loss 1.4810 
step 3000: train loss 1.1397 , val_loss 1.4862 
step 3250: train loss 1.1073 , val_loss 1.4732 
step 3500: train loss 1.0877 , val_loss 1.4813 
step 3750: train loss 1.0549 , val_loss 1.4978 
step 4000: train loss 1.0293 , val_loss 1.5005 
step 4250: train loss 1.0072 , val_loss 1.5174 
step 4500: train loss 0.9803 , val_loss 1.5144 
step 4750: train loss 0.9511 , val_loss 1.5287 
step 5000: trai

In [ ]:
from torch.cuda import temperature
#generate shakesphere output
print("\n---generating sample test---\n")
# Instead of starting with zeros:
prompt = "KING RICHARD:\nWhat say you, "
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

generated_ids = model.generate(context, max_new_tokens=500, temperature=0.6, top_k=20)[0].tolist()
print(decode(generated_ids))



---generating sample test---

KING RICHARD:
What say you, or I,
That so think you do that play him in my soul
And not move it to take itself?

Surse:
My master, to my fay to the County.

CATESBY:
My lord, to-morrow, the Signior Margaret
Which the consent he is.

HASTINGS:
Far not the matter
To England's Christend; set on thy night,
The cause the means the gods find the trial of the worst.

CATESBY:
My lord,
My lord, your son was an impediment.

KING RICHARD III:
I would so well I?

CATESBY:
My lord.

KING RICHARD III:
My lord, this is so he well.

KING


In [ ]:
from google.colab import files

# Save weights to file
torch.save(model.state_dict(), "nanogpt_weights.pt")

# Download to your computer
files.download("nanogpt_weights.pt")

NameError: name 'torch' is not defined

In [ ]:
#next time loading the weiths

In [ ]:
# 1. Instantiate the exact same architecture
model = NanoGPT(vocab_size, n_embd, n_head, n_layer, block_size, dropout).to(device)

# 2. Load the downloaded weights file
model.load_state_dict(torch.load("nanogpt_shakespeare.pt", map_location=device))
model.eval()

# 3. Generate text immediately!
prompt = "KING RICHARD:\n"
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=500, temperature=0.7, top_k=20)
print(decode(generated[0].tolist()))